# Лабораторная работа 4  
## Нейронная сеть для бинарной классификации без фреймворков

---

## Содержание

1. [Теоретическая часть](#1-теоретическая-часть)
2. [Описание реализации](#2-описание-реализации)
3. [Эксперименты и результаты](#3-эксперименты-и-результаты)
4. [Выводы](#4-выводы)

---

## 1. Теоретическая часть

### 1.1 Перцептрон и прямой проход

Один нейрон вычисляет скалярное произведение входа на вектор весов, добавляет смещение и применяет нелинейную функцию активации:

$$z = \mathbf{w}^\top \mathbf{x} + b, \qquad a = f(z)$$

Для батча из $N$ примеров операция записывается матрично:

$$\mathbf{Z} = \mathbf{X} \mathbf{W} + \mathbf{b}, \quad \mathbf{X} \in \mathbb{R}^{N \times d_{\text{in}}}, \; \mathbf{W} \in \mathbb{R}^{d_{\text{in}} \times d_{\text{out}}}$$

В работе реализованы две архитектуры:

**Однослойная сеть** (без скрытых слоёв, аналог логистической регрессии):

$$\hat{y} = \sigma(\mathbf{x} \mathbf{W}_1 + \mathbf{b}_1)$$

**Перцептрон с одним скрытым слоем:**

$$\mathbf{h} = \text{BReLU}(\mathbf{x} \mathbf{W}_1 + \mathbf{b}_1), \qquad \hat{y} = \sigma(\mathbf{h} \mathbf{W}_2 + \mathbf{b}_2)$$

---

### 1.2 Функции активации

**Sigmoid** используется на выходном слое — отображает $\mathbb{R} \to (0, 1)$ и интерпретируется как вероятность принадлежности к классу 1:

$$\sigma(x) = \frac{1}{1 + e^{-x}}, \qquad \sigma'(x) = \sigma(x)(1 - \sigma(x))$$

**BReLU (Bipolar ReLU)** — абсолютно-значная версия, используется в скрытом слое:

$$\text{BReLU}(x) = |x|, \qquad \frac{d}{dx}|x| = \text{sign}(x)$$

Преимущества BReLU перед классической ReLU:

| Свойство | ReLU | BReLU |
|---|---|---|
| Мёртвые нейроны ($a=0$ при $x<0$) | Да | Нет |
| Симметрия | Нет | $f(-x)=f(x)$ |
| Ненулевой градиент | Только при $x>0$ | При всех $x \neq 0$ |
| Насыщение | Нет (при $x>0$) | Нет |

Каждый нейрон скрытого слоя реализует V-образную функцию $|\mathbf{w}^\top \mathbf{x} + b|$, что позволяет кодировать двустороннюю реакцию на отклонение от гиперплоскости.

---

### 1.3 Функция потерь: Binary Cross-Entropy

$$\mathcal{L} = -\frac{1}{N} \sum_{i=1}^{N} \left[ y_i \log \hat{y}_i + (1 - y_i) \log(1 - \hat{y}_i) \right]$$

Это отрицательное логарифмическое правдоподобие модели Бернулли. Градиент по предсказанию:

$$\frac{\partial \mathcal{L}}{\partial \hat{y}_i} = -\frac{y_i}{\hat{y}_i} + \frac{1 - y_i}{1 - \hat{y}_i}$$

При совместном использовании BCE и Sigmoid градиент по входу сигмоиды $z$ упрощается:

$$\frac{\partial \mathcal{L}}{\partial z} = \frac{\hat{y} - y}{N}$$

---

### 1.4 Обратное распространение ошибки

**Chain rule** — основа алгоритма backpropagation. Для линейного слоя $\mathbf{Z} = \mathbf{X}\mathbf{W} + \mathbf{b}$, если $\delta = \partial \mathcal{L} / \partial \mathbf{Z}$ — градиент, поступающий «сверху»:

$$\frac{\partial \mathcal{L}}{\partial \mathbf{W}} = \mathbf{X}^\top \delta \quad (d_{\text{in}} \times d_{\text{out}})$$

$$\frac{\partial \mathcal{L}}{\partial \mathbf{b}} = \mathbf{1}^\top \delta \quad (1 \times d_{\text{out}})$$

$$\frac{\partial \mathcal{L}}{\partial \mathbf{X}} = \delta \mathbf{W}^\top \quad (N \times d_{\text{in}})$$

Последнее выражение передаётся ниже по цепочке слоёв.

---

### 1.5 SGD и mini-batch

| Режим | batch_size | Свойства |
|---|---|---|
| Full GD | $N$ | Точный градиент, дорого при большом $N$ |
| True SGD | $1$ | Шумный градиент, быстрые обновления, нестабилен |
| Mini-batch SGD | $1 < B < N$ | Компромисс: шум помогает избегать локальных минимумов |

Обновление mini-batch:

$$\theta_{t+1} = \theta_t - \alpha \cdot \hat{g}_t, \quad \hat{g}_t = \frac{1}{B} \sum_{i \in \mathcal{B}_t} \nabla_\theta \mathcal{L}_i$$

---

### 1.6 Оптимизаторы

#### Momentum

Накапливает «скорость» в направлении устойчивого градиента, демпфируя осцилляции:

$$v_t = \beta_1 v_{t-1} + (1 - \beta_1) g_t, \qquad \theta_{t+1} = \theta_t - \alpha v_t$$

#### Nesterov Accelerated Gradient (NAG)

«Смотрит вперёд»: вычисляет градиент в точке, куда momentum уже перенесёт параметры:

$$\theta_{\text{look}} = \theta_t - \alpha \beta_1 v_{t-1}, \qquad v_t = \beta_1 v_{t-1} + g(\theta_{\text{look}})$$

NAG тормозит перед минимумом, что уменьшает перелёт по сравнению с обычным momentum.

#### AdaGrad

Адаптивный learning rate: редко встречающимся признакам — большой шаг:

$$G_t = G_{t-1} + g_t^2, \qquad \theta_{t+1} = \theta_t - \frac{\alpha}{\sqrt{G_t + \varepsilon}} g_t$$

**Недостаток:** $G_t$ монотонно растёт, и $\alpha_{\text{eff}} \to 0$ на длинных прогонах.

#### RMSProp

Устраняет проблему AdaGrad за счёт экспоненциального скользящего среднего:

$$s_t = \rho s_{t-1} + (1 - \rho) g_t^2, \qquad \theta_{t+1} = \theta_t - \frac{\alpha}{\sqrt{s_t + \varepsilon}} g_t$$

#### Adam = Momentum + RMSProp + Bias Correction

$$m_t = \beta_1 m_{t-1} + (1 - \beta_1) g_t \quad \text{(1-й момент)}$$
$$v_t = \beta_2 v_{t-1} + (1 - \beta_2) g_t^2 \quad \text{(2-й момент)}$$
$$\hat{m}_t = \frac{m_t}{1 - \beta_1^t}, \quad \hat{v}_t = \frac{v_t}{1 - \beta_2^t} \quad \text{(коррекция начального смещения)}$$
$$\theta_{t+1} = \theta_t - \alpha \frac{\hat{m}_t}{\sqrt{\hat{v}_t} + \varepsilon}$$

Bias correction компенсирует тот факт, что $m_0 = v_0 = 0$, из-за чего первые шаги без коррекции были бы занижены. Типичные гиперпараметры: $\beta_1 = 0.9$, $\beta_2 = 0.999$, $\varepsilon = 10^{-8}$, $\alpha = 10^{-3}$.

---

### 1.7 Инициализация He и регуляризация

Веса инициализируются методом He:

$$\mathbf{W} \sim \mathcal{N}\!\left(0,\, \sqrt{\tfrac{2}{d_{\text{in}}}}\right)$$

Это поддерживает дисперсию активаций примерно постоянной при прохождении через слои с ReLU-подобными активациями.

**L2-регуляризация (weight decay)** добавляет к градиенту штраф $\lambda \theta$:

$$g_t \leftarrow g_t + \lambda \theta_t$$

что эквивалентно минимизации $\mathcal{L} + \frac{\lambda}{2}\|\theta\|^2$ и предотвращает переобучение при малых датасетах.

---

## 2. Описание реализации

### 2.1 Структура проекта

```
lab4.ipynb          — эксперименты и визуализация
src/
  activations.py   — BReLU (|x|) и Sigmoid с forward/backward
  layers.py        — Linear: z = XW + b, He-инициализация
  losses.py        — BinaryCrossEntropy
  optimizers.py    — AdamOptimizer (Momentum/Nesterov × RMSProp/AdaGrad) + SGD
  models.py        — SingleLayerNet, OneHiddenLayerNet
  trainer.py       — цикл mini-batch SGD + Early Stopping + snapshot весов
  data_utils.py    — train/val/test split, StandardScaler
  visualization.py — кривые обучения, граница решения, метрики, матрица ошибок
```

### 2.2 Линейный слой (`layers.py`)

Реализует $\mathbf{Z} = \mathbf{X}\mathbf{W} + \mathbf{b}$.

```python
def forward(self, X):
    self.input = X
    return X @ self.W + self.b

def backward(self, grad_output):
    self.dW = self.input.T @ grad_output   # ∂L/∂W
    self.db = grad_output.sum(axis=0, keepdims=True)  # ∂L/∂b
    return grad_output @ self.W.T          # ∂L/∂X — передаётся вниз
```

Ключевой момент: `self.input` сохраняется во время `forward` и используется в `backward` — это стандартный паттерн вычислительного графа.

### 2.3 BReLU (`activations.py`)

```python
def forward(self, x):
    self.input = x
    return np.abs(x)

def backward(self, grad_output):
    return grad_output * np.sign(self.input)  # субградиент в 0 = 0
```

### 2.4 Оптимизатор (`optimizers.py`)

Единый класс `AdamOptimizer` параметризован двумя флагами:

```python
AdamOptimizer(
    lr=1e-3,
    momentum_type='nesterov',  # 'momentum' | 'nesterov'
    adaptive_type='rmsprop',   # 'rmsprop'  | 'adagrad'
)
```

Это позволяет в рамках одного класса получить четыре конфигурации и сравнить их в едином эксперименте. Состояния первого и второго момента хранятся в словарях, индексированных по `id(param)`, что делает оптимизатор независимым от конкретной архитектуры модели.

### 2.5 Тренер (`trainer.py`)

Цикл обучения реализует:

1. **Mini-batch SGD:** перемешивание индексов в начале каждой эпохи, нарезка на батчи размера `batch_size`.
2. **Early Stopping:** счётчик `_no_improve_count` инкрементируется, когда `val_loss` не улучшается более чем на `min_delta`. При превышении `patience` — остановка.
3. **Snapshot весов:** при каждом улучшении `val_loss` создаётся `param.copy()` для всех параметров. По завершении обучения лучшие веса восстанавливаются in-place.

### 2.6 Разбиение данных и стандартизация (`data_utils.py`)

Разбиение 60/20/20 реализовано через перестановку индексов с фиксированным `random_state`. `StandardScaler` обучается **только на тренировочных данных** во избежание утечки информации:

```python
scaler.fit(X_train)          # μ и σ вычисляются по train
X_train_s = scaler.transform(X_train)
X_val_s   = scaler.transform(X_val)   # используем те же μ, σ
X_test_s  = scaler.transform(X_test)
```

---

## 3. Эксперименты и результаты

### 3.1 Датасеты

| Параметр | make_moons | make_classification |
|---|---|---|
| Всего примеров | 400 | 200 |
| Признаков | 2 | 5 (2 инф., 2 избыт., 1 шум) |
| Баланс классов | 50/50 | 50/50 |
| Train / Val / Test | 240 / 80 / 80 | 120 / 40 / 40 |
| Характер | Нелинейный (полумесяцы) | Линейно-разделимый |

### 3.2 Dataset 1: make_moons

#### Однослойная сеть

Гиперпараметры: lr=5e-3, batch_size=32, weight_decay=1e-4, Adam (Momentum+RMSProp), epochs=300, patience=30.

| Эпоха | Train loss | Train acc | Val loss | Val acc |
|---|---|---|---|---|
| 50 | 0.3190 | 0.875 | 0.2848 | 0.8625 |
| 100 | 0.2985 | 0.871 | 0.2552 | 0.8875 |
| 200 | 0.2931 | 0.891 | 0.2456 | 0.8875 |
| 300 | 0.2933 | 0.891 | 0.2448 | 0.8875 |

**Тест:** Accuracy = 0.85, Precision = 0.838, Recall = 0.838, F1 = 0.838  
Матрица ошибок: TP=31, TN=37, FP=6, FN=6.

Кривая обучения сошлась к эпохе ≈100 и далее стагнирует — модель упёрлась в предел своей выразительной способности. Линейная граница не может разделить два переплетённых полумесяца.

#### Сравнение конфигураций оптимизатора

Все четыре конфигурации дали схожую финальную val_loss ≈ 0.245:

| Конфигурация | Val loss (финал) | Val acc (финал) |
|---|---|---|
| Adam (Mom+RMS) | ≈0.245 | ≈0.888 |
| Nadam (Nest+RMS) | ≈0.245 | ≈0.888 |
| Mom+AdaGrad | ≈0.247 | ≈0.875 |
| Nest+AdaGrad | ≈0.247 | ≈0.875 |

RMSProp-варианты сходятся быстрее первые 50–100 эпох; AdaGrad-варианты замедляются в конце из-за монотонного роста $G_t$. Разница Nesterov vs Momentum минимальна — однослойная задача слишком проста для look-ahead.

#### Подбор hidden_size для OneHiddenLayer

| hidden | best val_loss | best val_acc |
|---|---|---|
| 4 | 0.2147 | 0.9000 |
| 8 | 0.2298 | 0.8875 |
| 16 | 0.0912 | 0.9875 |
| **32** | **0.0682** | **0.9875** |

h=8 хуже h=4 по val_loss — нерегулярность ландшафта потерь при слабой ёмкости сети. Резкий скачок качества происходит между h=8 и h=16: начиная с 16 нейронов сеть приобретает достаточно «степеней свободы» для аппроксимации нелинейной границы.

#### OneHiddenLayer (h=32)

Гиперпараметры: lr=1e-3, batch_size=32, weight_decay=1e-4, Nadam+RMSProp, epochs=500, patience=40.

| Эпоха | Train loss | Train acc | Val loss | Val acc |
|---|---|---|---|---|
| 100 | 0.2745 | 0.895 | 0.2621 | 0.925 |
| 200 | 0.1942 | 0.910 | 0.1752 | 0.950 |
| 300 | 0.1282 | 0.957 | 0.1226 | 0.963 |
| 400 | 0.0920 | 0.973 | 0.0871 | 0.988 |
| 500 | 0.0684 | 0.977 | 0.0682 | 0.988 |

**Тест:** Accuracy = 1.000, Precision = 1.000, Recall = 1.000, F1 = 1.000  
Матрица ошибок: TP=37, TN=43, FP=0, FN=0.

Train_loss и val_loss убывают синхронно весь прогон — признак хорошей генерализации. Зазор между train (0.068) и val (0.068) практически нулевой.

#### Сравнение архитектур на make_moons

| Модель | Test Acc | Test F1 |
|---|---|---|
| SingleLayer | 0.850 | 0.838 |
| OneHidden(h=32) | **1.000** | **1.000** |

---

### 3.3 Dataset 2: make_classification

#### Однослойная сеть

Гиперпараметры: lr=1e-3, batch_size=16, weight_decay=1e-3, Adam, epochs=400, patience=40.

| Эпоха | Train loss | Train acc | Val loss | Val acc |
|---|---|---|---|---|
| 50 | 0.3621 | 0.844 | 0.3494 | 0.850 |
| 100 | 0.3039 | 0.891 | 0.2839 | 0.900 |
| 300 | 0.2337 | 0.930 | 0.1980 | 0.900 |
| 400 | 0.2225 | 0.930 | 0.1843 | 0.900 |

**Тест:** Accuracy = 0.95, Precision = 0.941, Recall = 0.941, F1 = 0.941  
Матрица ошибок: TP=16, TN=22, FP=1, FN=1.

#### Влияние batch_size

Эксперимент с batch_size ∈ {1, 8, 16, 32, None} показал:

- **bs=1:** шумная кривая с осцилляциями — высокая дисперсия градиентов.
- **bs=8, 16:** наилучший баланс скорости и стабильности.
- **bs=32:** более гладкая, но медленнее сходящаяся кривая.
- **Full GD (bs=None):** 7–8 шагов оптимизатора за эпоху, самая медленная сходимость по числу эпох.

Финальная accuracy схожа у всех конфигураций — на малом датасете batch_size влияет на скорость сходимости, но не на итоговое качество.

#### Подбор hidden_size для OneHiddenLayer

| hidden | best val_loss | best val_acc |
|---|---|---|
| 4 | 0.1687 | 0.9000 |
| **8** | **0.1567** | **0.9750** |
| 16 | 0.1821 | 0.9500 |
| 32 | 0.1888 | 0.9000 |

Чёткая U-образная зависимость: h=4 — недообучение, h=8 — оптимум, h≥16 — переобучение. Сеть с h=32 имеет ≈225 параметров при 120 обучающих примерах — сильный дисбаланс.

#### OneHiddenLayer (h=8)

Гиперпараметры: lr=1e-3, batch_size=16, weight_decay=1e-3, Nadam+RMSProp, epochs=400, patience=40.

| Эпоха | Train loss | Train acc | Val loss | Val acc |
|---|---|---|---|---|
| 50 | 0.4708 | 0.906 | 0.3971 | 0.975 |
| 100 | 0.2632 | 0.930 | 0.2343 | 0.900 |
| 400 | 0.1636 | 0.930 | 0.1572 | 0.900 |

**Тест:** Accuracy = 0.95, Precision = 0.895, Recall = 1.000, F1 = 0.944  
Матрица ошибок: TP=17, TN=21, FP=2, FN=0.

Асимметрия ошибок: нет пропущенных положительных примеров (FN=0), но 2 ложных срабатывания (FP=2). Нелинейная граница с BReLU «расширяет» область класса 1.

#### Сравнение архитектур на make_classification

| Модель | Test Acc | F1 | Precision | Recall |
|---|---|---|---|---|
| SingleLayer | 0.950 | 0.941 | 0.941 | 0.941 |
| OneHidden(h=8) | 0.950 | **0.944** | 0.895 | **1.000** |

---

### 3.4 Итоговое сравнение всех четырёх моделей

| Датасет | Модель | Test Acc | Test F1 | Precision | Recall |
|---|---|---|---|---|---|
| make_moons | SingleLayer | 0.850 | 0.838 | 0.838 | 0.838 |
| make_moons | OneHidden(h=32) | **1.000** | **1.000** | 1.000 | 1.000 |
| make_classification | SingleLayer | 0.950 | 0.941 | 0.941 | 0.941 |
| make_classification | OneHidden(h=8) | 0.950 | **0.944** | 0.895 | **1.000** |

---

## 4. Выводы

### 4.1 Нелинейность задачи определяет выбор архитектуры

На **make_moons** (нелинейно разделимые данные) добавление скрытого слоя с BReLU критически необходимо: прирост составил +15 п.п. по accuracy (0.85 → 1.00). Однослойная сеть, эквивалентная логистической регрессии, не может аппроксимировать нелинейную границу — её предел определён классом линейных функций.

На **make_classification** (линейно-разделимые данные) прирост от скрытого слоя минимален: F1 улучшился лишь на 0.003 (0.941 → 0.944). Здесь однослойная сеть — сильная и более экономная базовая линия.

### 4.2 BReLU как активация скрытого слоя

BReLU(x) = |x| продемонстрировал устойчивость обучения: отсутствие «мёртвых нейронов» позволило сети с h=32 стабильно обучаться 500 эпох без деградации градиентов. Каждый нейрон реализует V-образную функцию, линейная комбинация которых может аппроксимировать произвольную непрерывную границу. Это согласуется с **теоремой об универсальной аппроксимации**: сеть с одним скрытым слоем и нелинейной активацией при достаточной ширине аппроксимирует любую непрерывную функцию на компакте.

### 4.3 Выбор лучшей модели через val_loss

Критерий отбора архитектуры — минимум `val_loss`. На make_moons это указало на h=32 (val_loss=0.068), на make_classification — на h=8 (val_loss=0.157). На малых датасетах **ёмкость модели должна соответствовать объёму обучающих данных**: h=32 при 120 примерах привело к переобучению (val_loss=0.189 > 0.157 при h=8).

Early Stopping с восстановлением лучших весов — эффективная встроенная регуляризация: он предотвращает переобучение без изменения архитектуры.

### 4.4 Сравнение оптимизаторов

Nadam (Nesterov + RMSProp) показал наибольшую скорость начальной сходимости среди четырёх конфигураций, что делает его рекомендуемым выбором. AdaGrad-варианты уступают на длинных прогонах из-за монотонного накопления $G_t$. Разница между Momentum и Nesterov мала на рассматриваемых датасетах — эффект look-ahead проявляется сильнее на задачах с шумным градиентом или в глубоких сетях.

### 4.5 Влияние batch_size

Оптимальный batch_size для датасетов в 120–240 примеров — **16–32**. True SGD (bs=1) нестабилен; Full GD медленно накапливает число шагов оптимизатора. Практическое правило: bs ≈ $\sqrt{N_{\text{train}}}$ даёт хороший компромисс.

### 4.6 Стандартизация признаков

Обучение `StandardScaler` исключительно на тренировочных данных — обязательное условие корректности. Нарушение этого правила приведёт к **утечке информации из val/test** и завышению метрик.

